In [1]:
import pandas as pd

## Creating one dataset for the control variables

Direct dependents, direct dependencies, download counts, and number of issues opened are all unique to each release.

For repository age , we have the date of the first release and need to calculate the repository age at each release date. 

**COME BACK AND ADD IN THE ISSUES OPENED DATA**

In [2]:
dependents = pd.read_csv('direct_dependents/dependent_count.csv')
dependencies = pd.read_csv('direct_dependencies/dependencies.csv')
download_counts = pd.read_csv('download_count/download_count.csv')
repo_age = pd.read_csv('repo_age/first_releases.csv')

In [3]:
package_data = pd.read_csv('../package_data.csv')
control_data = package_data[['project_name','package_name','tag_name','published_at','System']].copy()

## Need to add a column with the 'clean' version name to each dataframe

### **Step 1:** Removing any characters before the version numbers

In [4]:
import re

def clean_version(tag_name):
    """Remove package name prefixes before version numbers."""
    tag_name = str(tag_name)
    
    # Strategy 1: Find @ followed by a proper version pattern (digit.digit...)
    # Handles: @org/package@1.0.0 -> 1.0.0
    matches = list(re.finditer(r'@(\d+\.\d+[^\s]*)$', tag_name))
    if matches:
        return matches[-1].group(1)
    
    # Strategy 2: Find / followed by version pattern (for package/version style)
    # Handles: package-name/v1.0.0 -> 1.0.0
    match = re.search(r'/[v]?(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 3: Find _ followed by v and version pattern
    # Handles: gmv3_v2.0.0 -> 2.0.0
    match = re.search(r'_v(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 4: Handle package-version style with hyphen
    # Handles: package-1.0.0 -> 1.0.0
    match = re.search(r'-(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 5: Fallback - remove any non-digit prefix (for v1.0.0 style)
    # Handles: v1.0.0 -> 1.0.0
    match = re.search(r'^[^\d]*(\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    return tag_name

In [5]:
# Apply the clean_version function
control_data['tag_name_clean'] = control_data['tag_name'].apply(clean_version)
dependencies['tag_name_clean'] = dependencies['version'].apply(clean_version)
dependents['tag_name_clean'] = dependents['version'].apply(clean_version)
download_counts['tag_name_clean'] = download_counts['version'].apply(clean_version)

### **Step 2:** Need to remove the releases that do not follow semantic versioning

In [6]:
def is_semantic_version_format(tag_name):
    """
    Check if tag follows semantic versioning format:
    - number.number.number (e.g., 1.2.3)
    - number.number (e.g., 1.2)
    - number (e.g., 1)
    
    Does NOT allow any suffixes or prefixes.
    """
    tag_name = str(tag_name)
    
    # Strict pattern: ONLY digits and dots, nothing else
    # Must be: X, X.Y, or X.Y.Z where X, Y, Z are numbers
    pattern = r'^\d+(\.\d+)?(\.\d+)?$'
    
    return bool(re.match(pattern, tag_name))

## Uncomment and run the cell below to examine the releases that do not follow semantic versioning

In [ ]:
# Find all tags that don't follow the basic format
non_basic_format_package = control_data[~control_data['tag_name_clean'].apply(is_semantic_version_format)]
non_basic_format_dependency = dependencies[~dependencies['tag_name_clean'].apply(is_semantic_version_format)]
non_basic_format_dependent = dependents[~dependents['tag_name_clean'].apply(is_semantic_version_format)]
non_basic_format_download_counts = download_counts[~download_counts['tag_name_clean'].apply(is_semantic_version_format)]

# Show all unique non-basic format tags
print("*****************************************************************************")
print(f"\nAll non-basic PACKAGE_DATA format tags ({len(non_basic_format_package['tag_name_clean'].unique())} unique):")
print(non_basic_format_package['tag_name_clean'].head(100))

print("*****************************************************************************")
print(f"\nAll non-basic DEPENDENCY format tags ({len(non_basic_format_dependency['tag_name_clean'].unique())} unique):")
print(non_basic_format_dependency['tag_name_clean'].head(100))

print("*****************************************************************************")
print(f"\nAll non-basic DEPENDENCY format tags ({len(non_basic_format_dependent['tag_name_clean'].unique())} unique):")
print(non_basic_format_dependent['tag_name_clean'].head(100))

print("*****************************************************************************")
print(f"\nAll non-basic DOWNLOAD_COUNTS format tags ({len(non_basic_format_download_counts['tag_name_clean'].unique())} unique):")
print(non_basic_format_download_counts['tag_name_clean'].head(100))

In [8]:
## remove the non-basic format tags from each dataset
control_data = control_data[control_data['tag_name_clean'].apply(is_semantic_version_format)]
dependencies = dependencies[dependencies['tag_name_clean'].apply(is_semantic_version_format)]
dependents = dependents[dependents['tag_name_clean'].apply(is_semantic_version_format)]
download_counts = download_counts[download_counts['tag_name_clean'].apply(is_semantic_version_format)]

## Creating the dataset by merging one control variable at a time

**Starting with repository first release date**

In [ ]:
control_data['System'] = control_data['System'].str.lower()
repo_age['platform'] = repo_age['platform'].str.lower()

control_data = control_data.merge(repo_age, 
                                  left_on =['package_name','System'],
                                  right_on = ['package','platform'], 
                                  how='left')

control_data = control_data.drop(columns=['package', 'platform'])

In [20]:
## Filling in release date for missing values
control_data.loc[control_data['project_name'] == 'jupyter/jupyterlab', 'first_release'] = '2017-06-19' #https://github.com/jupyterlab/jupyterlab/releases?page=26
control_data.loc[control_data['project_name'] == 'projen/projen', 'first_release'] = '2021-04-21' #https://github.com/projen/projen/releases?page=164
control_data.loc[control_data['project_name'] == 'cdk8s-team/cdk8s-core', 'first_release'] = '2021-05-31' #https://github.com/cdk8s-team/cdk8s-core/releases?page=172
control_data.loc[control_data['project_name'] == 'cdklabs/cdk-ecr-deployment', 'first_release'] = '2021-05-03' #https://github.com/cdklabs/cdk-ecr-deployment/releases?page=33
control_data.loc[control_data['project_name'] == 'dhensby/node-http-message-signatures', 'first_release'] = '2021-12-10' #https://github.com/dhensby/node-http-message-signatures/releases
control_data.loc[control_data['project_name'] == 'kitware/wslink', 'first_release'] = '2017-07-21' #https://github.com/kitware/wslink/releases?page=8
control_data.loc[control_data['project_name'] == 'ccxt/ccxt', 'first_release'] = '2023-07-01' #https://github.com/ccxt/ccxt/releases?page=21
control_data.loc[control_data['project_name'] == 'hashberg-io/multiformats', 'first_release'] = '2021-12-16' #https://github.com/hashberg-io/multiformats/releases
control_data.loc[control_data['project_name'] == 'cdklabs/cdk-monitoring-constructs', 'first_release'] = '2022-02-24' #https://github.com/cdklabs/cdk-monitoring-constructs/releases?page=31
control_data.loc[control_data['project_name'] == 'replicate/replicate-python', 'first_release'] = '2023-03-03' #https://github.com/replicate/replicate-python/releases?page=6
control_data.loc[control_data['project_name'] == 'jupyterlab-contrib/jupyterlab-tour', 'first_release'] = '2020-07-08' #https://github.com/jupyterlab-contrib/jupyterlab-tour/releases?page=2
control_data.loc[control_data['project_name'] == 'convertapi/convertapi-node', 'first_release'] = '2023-09-27' #https://github.com/ConvertAPI/convertapi-library-node/releases
control_data.loc[control_data['project_name'] == 'microsoft/pyright', 'first_release'] = '2019-03-22' #https://github.com/microsoft/pyright/releases?page=49

In [21]:
control_data_test1 = control_data.merge(dependents, 
                                       left_on =['package_name','tag_name_clean'],
                                       right_on = ['package','tag_name_clean'], 
                                       how='left')
control_data_test1 = control_data_test1.drop(columns=['package', 'ecosystem', 'version'])

control_data_test2 = control_data_test1.merge(dependencies, 
                                       left_on =['package_name','tag_name_clean'],
                                       right_on = ['package','tag_name_clean'], 
                                       how='left')
control_data_test2 = control_data_test2.drop(columns=['package', 'ecosystem', 'version'])

control_data_test3 = control_data_test2.merge(download_counts, 
                                       left_on =['package_name','tag_name_clean'],
                                       right_on = ['package_name','tag_name_clean'], 
                                       how='left')

In [23]:
## remove rows in control_data_test3 where dependents, dependents, and download_counts are all NaN
control_data_test3_removed_na = control_data_test3[~(control_data_test3['dependents'].isna() & control_data_test3['dependencies'].isna() & control_data_test3['downloads_7_day_total'].isna())]

In [26]:
## what is we remove all of the NA values in control_data_test3?
control_data_test3_all_na_removed = control_data_test3[~control_data_test3.isna().any(axis=1)]

In [ ]:
# del issue_count_merged_1
# del issue_count_merged_2
# del issue_count_merged_11
# del issue_count_merged_22

In [15]:
## Format the issue data 
issues_opened_1 = pd.read_csv('issues_opened/test_unsure')
issues_opened_2 = pd.read_csv('issues_opened/updated_issues_opened_case_sensitive_keeping_null_values')

## step 1: remove the characters preceding the version number
issues_opened_1['tag_name_clean'] = issues_opened_1['version'].apply(clean_version)
issues_opened_2['tag_name_clean'] = issues_opened_2['version'].apply(clean_version)

## step 2: only keep semantic version formatted tags
issues_opened_1 = issues_opened_1[issues_opened_1['tag_name_clean'].apply(is_semantic_version_format)]
issues_opened_2 = issues_opened_2[issues_opened_2['tag_name_clean'].apply(is_semantic_version_format)]

In [16]:
## step 3: merge with control_data_test3
issue_count_merged_1 = control_data_test3.merge(issues_opened_1, 
                                       left_on =['project_name','tag_name_clean'],
                                       right_on = ['repo_name','tag_name_clean'], 
                                       how='left')

issue_count_merged_2 = control_data_test3.merge(issues_opened_2, 
                                       left_on =['project_name','tag_name_clean'],
                                       right_on = ['original_name','tag_name_clean'], 
                                       how='left')

In [24]:
# 1. Clean and Deduplicate Dataset 1
issues_1_unique = (issues_opened_1
    .groupby(['repo_name', 'tag_name_clean'])['issues_opened']
    .sum()
    .reset_index())

# 2. Clean and Deduplicate Dataset 2
issues_2_unique = (issues_opened_2
    .groupby(['original_name', 'tag_name_clean'])['issues_opened']
    .sum()
    .reset_index())

# 3. Perform the Merges with the Unique Data
issue_count_merged_1 = control_data_test3_removed_na.merge(
    issues_1_unique, 
    left_on=['project_name', 'tag_name_clean'],
    right_on=['repo_name', 'tag_name_clean'], 
    how='left'
)

issue_count_merged_2 = control_data_test3_removed_na.merge(
    issues_2_unique, 
    left_on=['project_name', 'tag_name_clean'],
    right_on=['original_name', 'tag_name_clean'], 
    how='left'
)

In [25]:
# Check for duplicates in your right-side tables
print(issues_opened_1.duplicated(subset=['repo_name', 'tag_name_clean']).sum())
print(issues_opened_2.duplicated(subset=['original_name', 'tag_name_clean']).sum())


# Check for duplicates - are the duplicates just because of the cleaned up version names?
print(issues_opened_1.duplicated(subset=['repo_name', 'version']).sum())
print(issues_opened_2.duplicated(subset=['original_name', 'version']).sum())

19463
23974
0
4511


# What if we join on the 'uncleaned' version numbers?
This results in almost 3x the number of missing values.

In [19]:
control_data_test_unclean = control_data.merge(dependents, 
                                       left_on =['package_name','tag_name'],
                                       right_on = ['package','version'], 
                                       how='left')